# Review Intelligence — Colab DistilBERT training

**Runtime → Change runtime type → GPU (T4)**

This notebook:
1. Installs deps
2. Streams Amazon Electronics reviews and builds stratified splits
3. Fine-tunes `distilbert-base-uncased` for 3-class sentiment
4. Saves `best-model/` for download back to your laptop

After training, follow `COLAB_EXPORT.md` on your local machine.

## 1) Install packages

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas mlflow pyarrow
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2) Config

Use `MAX_SAMPLES=50000` for a serious run. Start with `5000` for a quick Colab smoke test.

In [ ]:
from pathlib import Path

MAX_SAMPLES = 50000          # lower to 5000 for a fast smoke run
NUM_EPOCHS = 3
TRAIN_BATCH = 16
EVAL_BATCH = 32
MAX_LENGTH = 256
MODEL_NAME = 'distilbert-base-uncased'
OUTPUT_DIR = Path('/content/checkpoints/production-distilbert')
DATA_DIR = Path('/content/data/processed')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ELECTRONICS_URL = (
    'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/'
    'review_categories/Electronics.jsonl.gz'
)

## 3) Build dataset (or upload your local parquet)

Optional shortcut: upload `train.parquet` / `validation.parquet` / `test.parquet` from your laptop to `/content/data/processed/` and skip the next cell.

In [ ]:
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.model_selection import train_test_split

def map_rating_to_label(rating: float) -> int:
    if rating <= 2:
        return 0
    if rating == 3:
        return 1
    return 2

def coerce_text(example):
    text = str(example.get('text') or '').strip()
    title = str(example.get('title') or '').strip()
    return text or title

train_path = DATA_DIR / 'train.parquet'
if train_path.exists():
    print('Using uploaded parquet splits from', DATA_DIR)
else:
    print('Streaming Electronics reviews...')
    stream = load_dataset('json', data_files=ELECTRONICS_URL, split='train', streaming=True)
    rows = []
    for i, record in enumerate(stream):
        text = coerce_text(record)
        if not text:
            continue
        rating = float(record['rating'])
        rows.append({
            'text': text,
            'label': map_rating_to_label(rating),
            'rating': rating,
            'parent_asin': str(record.get('parent_asin', '')),
        })
        if len(rows) >= MAX_SAMPLES:
            break

    labels = [r['label'] for r in rows]
    train_rows, temp_rows = train_test_split(rows, test_size=0.2, random_state=42, stratify=labels)
    temp_labels = [r['label'] for r in temp_rows]
    val_rows, test_rows = train_test_split(temp_rows, test_size=0.5, random_state=42, stratify=temp_labels)

    splits = DatasetDict({
        'train': Dataset.from_list(train_rows),
        'validation': Dataset.from_list(val_rows),
        'test': Dataset.from_list(test_rows),
    })
    for name, ds in splits.items():
        ds.to_parquet(DATA_DIR / f'{name}.parquet')
        print(name, len(ds))

print('Ready:', list(DATA_DIR.glob('*.parquet')))

## 4) Train DistilBERT

In [ ]:
import json
import numpy as np
import torch
import mlflow
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

LABEL_NAMES = {0: 'negative', 1: 'neutral', 2: 'positive'}

files = {
    'train': str(DATA_DIR / 'train.parquet'),
    'validation': str(DATA_DIR / 'validation.parquet'),
    'test': str(DATA_DIR / 'test.parquet'),
}
splits = load_dataset('parquet', data_files=files)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized = splits.map(tokenize, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
keep = {'input_ids', 'attention_mask', 'labels'}
for split in tokenized:
    drop = [c for c in tokenized[split].column_names if c not in keep]
    if drop:
        tokenized[split] = tokenized[split].remove_columns(drop)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro')),
    }

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1_macro',
    greater_is_better=True,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

mlflow.set_tracking_uri('/content/mlruns')
mlflow.set_experiment('review-intelligence-colab')

with mlflow.start_run():
    mlflow.log_params({
        'model': MODEL_NAME,
        'max_samples': MAX_SAMPLES,
        'epochs': NUM_EPOCHS,
        'train_batch': TRAIN_BATCH,
    })
    trainer.train()
    val_metrics = trainer.evaluate(eval_dataset=tokenized['validation'])
    test_metrics = trainer.evaluate(eval_dataset=tokenized['test'], metric_key_prefix='test')
    mlflow.log_metrics({k: float(v) for k, v in {**val_metrics, **test_metrics}.items()})

    best_dir = OUTPUT_DIR / 'best-model'
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    print('Saved', best_dir)
    print('validation', val_metrics)
    print('test', test_metrics)

## 5) Zip and download checkpoint

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/best-model'
shutil.make_archive(zip_base, 'zip', root_dir=OUTPUT_DIR, base_dir='best-model')
print('Created', zip_base + '.zip')
files.download(zip_base + '.zip')

## Next on your laptop

Open `notebooks/COLAB_EXPORT.md` and:
1. Unzip into `project/checkpoints/production-distilbert/`
2. Relink `checkpoints/best-model`
3. Restart `uvicorn` and hit `/api/analyze`